# Решения: группы

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def find_titanic_csv() -> Path:
    for p in (Path('titanic.csv'), Path('../../data/titanic.csv'), Path('../data/titanic.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('titanic.csv не найден — положите файл рядом с ноутбуком')


TITANIC_PATH = find_titanic_csv()
df = pd.read_csv(TITANIC_PATH)

import matplotlib.pyplot as plt
from pathlib import Path as _P
_P('figures').mkdir(exist_ok=True)


## Урок

In [ ]:
plt.figure()
df.loc[df['survived']==0, 'fare'].hist(bins=30, alpha=0.5, label='0')
df.loc[df['survived']==1, 'fare'].hist(bins=30, alpha=0.5, label='1')
plt.legend(); plt.tight_layout()
fare_hist_path = _P('figures/fare_by_survived_hist.png'); plt.savefig(fare_hist_path); plt.close()
FARE_HIST_NOTE = 'у выживших чаще более высокий fare, но распределения перекрываются'
rates = df.groupby(['sex', 'pclass'])['survived'].mean().unstack()
plt.figure(); df.groupby('who')['survived'].mean().plot(kind='bar'); plt.tight_layout()
who_bar_path = _P('figures/survived_by_who.png'); plt.savefig(who_bar_path); plt.close()
rate_A = float(df.dropna(subset=['age'])['survived'].mean())
d2 = df.copy(); d2['age'] = d2['age'].fillna(d2['age'].median())
mean_age_surv_B = float(d2.loc[d2['survived']==1, 'age'].mean())
mean_age_dead_B = float(d2.loc[d2['survived']==0, 'age'].mean())
WHEN_A = 'Когда вопрос именно о доле survived без искажения импутацией возраста'
share_p1_all = float((df['pclass']==1).mean())
share_p1_deck = float((df.dropna(subset=['deck'])['pclass']==1).mean())
CAREFUL_CLAIM = 'Наблюдается ассоциация пола с выживаемостью; причинность из EDA не следует'
rows = []
for s in ('female', 'male'):
    g = df[df['sex']==s]
    rows.append({'group': f'sex={s}', 'n': len(g), 'survival_rate': g['survived'].mean()})
for c in (1, 2, 3):
    g = df[df['pclass']==c]
    rows.append({'group': f'pclass={c}', 'n': len(g), 'survival_rate': g['survived'].mean()})
summary = pd.DataFrame(rows)
by_town = df.dropna(subset=['embark_town']).groupby('embark_town')['survived'].mean()
TOWN_NOTE = 'Cherbourg выше — связать с долей 1 класса, не с магией города'
print(rates, summary)

## ДЗ

In [ ]:
plt.figure(); df.groupby('pclass')['survived'].mean().plot(kind='bar'); plt.tight_layout()
p = _P('figures/surv_by_pclass_hw.png'); plt.savefig(p); plt.close()
rate_f3 = float(df.loc[(df['sex']=='female')&(df['pclass']==3), 'survived'].mean())
rate_m1 = float(df.loc[(df['sex']=='male')&(df['pclass']==1), 'survived'].mean())
POLICY = (
    'В отчёте: доли survived — на полной таблице без импутации age. '
    'Описания возраста — отдельно на non-NA или с пометкой median fill. '
    'deck не используем как группу из-за огромной доли NA.'
)
rate_after = float(df.drop_duplicates()['survived'].mean())
CHANGED = abs(rate_after - df['survived'].mean()) > 1e-12
print(rate_f3, rate_m1, CHANGED)